# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Akash0-0/InternShip_Task01/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 1 — Ranking Signal Analysis** · warehouse slice: `fact_content_daily_performance` (mid-panel month **2026-03**).

What this notebook proves, in order:
1. the contract in plain words (one row, tables, window, label/proxy, exclusions),
2. three facts about my slice, verified with three real queries on the warehouse,
3. a five-feature frame for my lane, each feature knowable at the decision moment,
4. the leakage trap — one label-derived column makes the score jump, then I delete it and keep the honest number,
5. one named limitation of my slice.

> Skills applied: `writing-data-contracts` (methodology) + `flyrank/flyrank-data` (warehouse layout, grain, availability flags).


## 1. Unit of analysis + time window

**The contract, in plain words (5 answers):**

1. **One row means** — in `fact_content_daily_performance`, one row = **one content item (page) × one report date × one client**: a daily measurement row. My lane works at **page-month grain**: one row = one page's March 2026 search performance **plus** its April outcome.
2. **Which table(s)** — `fact_content_daily_performance` (features + label), joined to `dim_content` for context only (created date, content type, word count), and `dim_clients` to know each client's history coverage.
3. **Time window** — features from **2026-03-01 → 2026-03-31** (a mid-panel month); the label comes from the next month **2026-04-01 → 2026-04-30**. The full panel runs 2025-01-27 → 2026-06-30; the final month is **sealed** as the test month.
4. **What I predict / rank** — a proxy label: **`is_declining_next`** = did the page's GSC impressions drop ≥ 20% from March to April (`imp_apr < 0.8 × imp_mar`, with `imp_mar > 0`). It ranks pages for review priority.
5. **Deliberately excluded** — GA4 fields on rows where `ga4_data_available IS NOT TRUE` (a zero there means *not tracked*, not *no engagement*); raw query/URL/keyword text (private); product decision flags (not shipped, and circular); and the `_sample` table / final month 2026-06 (the natural outcome window of any past→future label — sealed).

Verify the setup and my slice's tables below, then §3 proves the three facts with real queries.

In [1]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"      # keep the executed notebook clean
import duckdb, pandas as pd, numpy as np

# Token: Colab secret HF_TOKEN, else the local huggingface cache. Never printed, never committed.
HF_TOKEN = os.environ.get("HF_TOKEN") or open(
    os.path.expanduser("~/.cache/huggingface/token")).read().strip()
assert HF_TOKEN, "Set the HF_TOKEN secret (Colab) or log in locally with huggingface-cli."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR  = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_APR  = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

# Context: how many clients are in the warehouse at all (dim_clients is small)
n_clients = con.sql(f"SELECT COUNT(*) FROM {DIM_CLIENTS}").fetchone()[0]
print("warehouse dim_clients:", n_clients)
print("slice: features 2026-03-01..31 | label 2026-04-01..30 | final month 2026-06 sealed")

warehouse dim_clients: 104
slice: features 2026-03-01..31 | label 2026-04-01..30 | final month 2026-06 sealed


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| **Features** (knowable by 2026-03-31) | `log_imp_mar`, `ctr_mar`, `avg_pos_mar`, `days_with_imp_mar`, `content_age_days` | built only from March daily rows + content metadata — nothing from April |
| **Label / proxy** (never a feature) | `is_declining_next` = `imp_apr < 0.8 × imp_mar` (April outcome) | the thing I rank for; defined on the *next* month |
| **Context** | `client_hash_id`, `content_hash_id` | grouping and joins only — anonymous hashes, never printed |
| **Excluded + why** | `imp_apr` (the label's own numerator — future information); GA4 fields where `ga4_data_available IS NOT TRUE`; raw query/URL/keyword text; product decision flags; `month=2026-06` sample | each one would smuggle the answer in, or is private, or is not tracked |

Build the page-month frame for my slice below (features from March, label from April).

In [2]:
# --- §2: page-month frame for my slice (features: March, label: April) ---

mar = con.sql(f"""
SELECT content_hash_id, client_hash_id,
       SUM(gsc_impressions)                                            AS imp_mar,
       SUM(gsc_clicks)                                                 AS clk_mar,
       AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END)   AS avg_pos_mar,
       SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END)            AS days_with_imp_mar
FROM {FACT_MAR}
GROUP BY 1, 2
""").df()

apr = con.sql(f"SELECT content_hash_id, SUM(gsc_impressions) AS imp_apr FROM {FACT_APR} GROUP BY 1").df()

dimc = con.sql(f"""
SELECT content_hash_id, content_created_date, content_type, word_count
FROM {DIM_CONTENT}
""").df()

frame = (mar.merge(apr, on="content_hash_id", how="left")
            .merge(dimc, on="content_hash_id", how="left"))
frame["imp_apr"] = frame["imp_apr"].fillna(0).astype(int)   # no April rows => zero observed impressions
frame["content_age_days"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(frame["content_created_date"])).dt.days

# the five features (max) + the label
frame["log_imp_mar"]       = np.log1p(frame["imp_mar"])
frame["ctr_mar"]           = 100.0 * frame["clk_mar"] / frame["imp_mar"].replace(0, np.nan)
frame["is_declining_next"] = ((frame["imp_mar"] > 0) & (frame["imp_apr"] < 0.8 * frame["imp_mar"])).astype(int)

FEATURES = ["log_imp_mar", "ctr_mar", "avg_pos_mar", "days_with_imp_mar", "content_age_days"]

print("frame:", frame.shape, "| label rate (all pages):", round(frame['is_declining_next'].mean(), 4))
print("pages with March impressions:", int((frame['imp_mar'] > 0).sum()))
frame[["content_hash_id", "client_hash_id"] + FEATURES + ["is_declining_next"]].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

frame: (331437, 14) | label rate (all pages): 0.2836
pages with March impressions: 176738


,content_hash_id,client_hash_id,log_imp_mar,ctr_mar,avg_pos_mar,days_with_imp_mar,content_age_days,is_declining_next
0,content_139be3b3532f1808,client_62f4a7e64f5e0096,6.597146,1.092896,2.147651,31.0,196,0
1,content_7c510d16d5e53a4b,client_62f4a7e64f5e0096,6.922644,0.098619,4.706891,31.0,196,1
2,content_46dfac88af1757e8,client_62f4a7e64f5e0096,7.957877,0.140007,1.845913,31.0,196,1
3,content_a61da9fdb2ac5659,client_62f4a7e64f5e0096,4.820282,0.000000,21.933642,12.0,195,1
4,content_5e0b9e6ed983db9a,client_62f4a7e64f5e0096,6.834109,0.107759,27.927554,31.0,195,1


## 3. Verify it with queries — three facts, three real queries

Every claim in §1 gets a query below, all on the same mid-panel month **2026-03** (the `_sample` table is the final month — a sealed test month, never used to develop labels).

**Fact 1 — the grain.** One row really is one `report_date × client × content`. If the grain claim is true, grouping by those three keys and asking for `COUNT(*) > 1` returns **0 rows**.

In [3]:
q_grain = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
FROM {FACT_MAR}
GROUP BY 1, 2, 3
HAVING n > 1
LIMIT 5
""").df()
print("rows that break the grain (expect 0):", len(q_grain))
q_grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows that break the grain (expect 0): 0


,report_date,client_hash_id,content_hash_id,n


**Fact 2 — my slice's row count and date span.** The whole 2026-03 partition of `fact_content_daily_performance`, plus how many distinct clients and pages it covers.

In [4]:
q_span = con.sql(f"""
SELECT COUNT(*) AS n_rows,
       MIN(report_date) AS first_date,
       MAX(report_date) AS last_date,
       COUNT(DISTINCT client_hash_id)  AS n_clients,
       COUNT(DISTINCT content_hash_id) AS n_contents
FROM {FACT_MAR}
""").df()
q_span

,n_rows,first_date,last_date,n_clients,n_contents
0,9841378,2026-03-01,2026-03-31,55,331437


**Fact 3 — availability.** The flag `ga4_data_available` can be `FALSE` or `NULL`; only `IS TRUE` rows have real GA4 numbers. Filter with `IS TRUE` and count how many rows survive — everything else stays in my slice but only for GSC-based signals.

In [5]:
q_avail = con.sql(f"""
SELECT COUNT(*) AS n_rows_total,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS n_rows_with_gsc,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS n_rows_with_ga4
FROM {FACT_MAR}
""").df()
q_avail

,n_rows_total,n_rows_with_gsc,n_rows_with_ga4
0,9841378,3611061,413966


### 3b. The five-feature frame — and when each feature is knowable

Decision moment = **2026-03-31 23:59**, when a content team would decide which pages to review. Every feature must be computable from data that exists at that moment.

| # | Feature | What it is | Knowable at the decision moment because… |
|---|---|---|---|
| 1 | `log_imp_mar` | log(1 + March GSC impressions) | every daily row is dated on or before 2026-03-31, and summing them is a pure-March computation |
| 2 | `ctr_mar` | March clicks ÷ impressions × 100 | both clicks and impressions come from the same March daily rows |
| 3 | `avg_pos_mar` | mean GSC position in March (days with a position) | it averages daily position values already recorded by 2026-03-31 |
| 4 | `days_with_imp_mar` | count of March days with ≥ 1 impression | it counts days strictly inside the March window |
| 5 | `content_age_days` | page age in days at 2026-03-31 | `content_created_date` is metadata known since the page was created, and the calendar date is the decision date itself |

The label (`is_declining_next`) is deliberately **not** in this list — it needs April data, so it is *not* knowable at the decision moment. That is exactly why it is the label.

In [6]:
frame[FEATURES].describe().T

,count,mean,std,min,25%,50%,75%,max
log_imp_mar,331437.0,2.670719,3.093188,0.000000,0.0,1.098612,5.379897,13.332827
ctr_mar,176738.0,0.459397,3.775992,0.000000,0.0,0.000000,0.215796,100.000000
avg_pos_mar,175304.0,17.050555,18.333942,0.101639,5.5,9.000000,22.000000,309.000000
days_with_imp_mar,331437.0,10.895166,13.197774,0.000000,0.0,1.000000,28.000000,31.000000
content_age_days,331437.0,207.795521,120.549978,-5.000000,103.0,220.000000,278.000000,494.000000


### 3c. The trap — one label-derived column, on purpose

Lesson from notebook 02: my label is computed from the margin `imp_apr < 0.8 × imp_mar` — so **that exact margin is the answer in disguise** (like `trend_pct` was for `trend_direction` in notebook 02). Feed it in as one feature and the quick score jumps toward perfect. Watch it, then delete it and keep the honest number.

**Honest quick score first** (pages with March visibility only):

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

mod = frame[frame["imp_mar"] > 0].copy()          # only pages that had some March visibility
X = mod[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
y = mod["is_declining_next"].values

def quick_score(X, y):
    m = LogisticRegression(max_iter=1000, random_state=42).fit(X, y)
    return roc_auc_score(y, m.predict_proba(X)[:, 1])

print("honest quick score (ROC-AUC):", round(quick_score(X, y), 4))

honest quick score (ROC-AUC): 0.592


Now add the **one label-derived column** — `leak_margin = imp_apr − 0.8 × imp_mar`, the exact margin the label is computed from (declining ⇔ margin < 0):

In [8]:
X_leaky = X.copy()
X_leaky["leak_margin"] = (mod["imp_apr"] - 0.8 * mod["imp_mar"]).values   # ← label-derived, added on purpose

print("leaky quick score (ROC-AUC):", round(quick_score(X_leaky, y), 4), " <- looks amazing")

# read the split, like notebook 02
from sklearn.tree import DecisionTreeClassifier, export_text
tree = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_leaky, y)
print()
print(export_text(tree, feature_names=list(X_leaky.columns)))

leaky quick score (ROC-AUC): 1.0  <- looks amazing

|--- leak_margin <= -0.10
|   |--- class: 1
|--- leak_margin >  -0.10
|   |--- class: 0



The tree splits straight on `leak_margin ≤ 0` and nails the label — because the label **is** `leak_margin < 0`. That is leakage: the feature is the answer in disguise and teaches nothing. Now delete the column and keep the honest number:

In [9]:
X_clean = X_leaky.drop(columns=["leak_margin"])
print("honest quick score after removing the leak:", round(quick_score(X_clean, y), 4))
print("(unchanged — the leak was the only thing that inflated it)")

honest quick score after removing the leak: 0.592
(unchanged — the leak was the only thing that inflated it)


## 4. Data limits

**One named limitation of my slice: the unbalanced panel.** My window is a global calendar month (2026-03), but clients joined the warehouse on different dates (`gsc_data_start` / `ga4_data_start` differ per client, some as late as April 2026) — so a page's "March" can be a full 31 days or a fraction of one, and pages registered mid-window get truncated, noisier features. Any ranking built from this frame inherits that uneven history.

Other honest limits:
- **GA4 is thin** — only ~4% of my slice's rows are `ga4_data_available IS TRUE`; on the rest, GA4 zeros mean *not tracked*, not *no engagement* (that's why GA4 fields are excluded from features).
- **Absence ≠ decline** — ~47% of March pages have no April rows at all; I read that as zero observed impressions, but it can also reflect tracking gaps, not organic decline.
- **The label is a proxy** — an impressions drop is not the same as a business outcome; seasonality and consolidation are not disentangled here.
- **Observational only** — this analysis describes associations in my slice; it cannot prove what causes a decline.



In [10]:
# No code needed here — the limits are claims about the data, not numbers to print.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only anonymous hashes
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.